# Inverse planning model

With reward and cost

In [1]:
from memo import memo
import jax
import jax.numpy as jnp

import numpy as np
import pandas as pd

from enum import IntEnum

## Model

In the most basic inverse planning model, participants observe an action, and evaluate what they think about how intimate the relationship is, on a scale from 0 to 100, where 0 is completely formal and 100 is completely intimate.

In [2]:
actions = jnp.array([0, 1, 2, 3])
IntimacyLevels = jnp.arange(0, 1.01, 0.01)

class RewardConditions(IntEnum):
    LOW = 0
    HIGH = 1

Get the risk for each of the 4 actions. Action 0 and 1 have no saliva transfer, so the risk is 0.

In [3]:
@jax.jit
def get_risk(action):
    return jnp.array([0, 0, 1, 2])[action]

Reward and cost

- In the low reward condition, the characters don't particularly want to eat the food together. So the reward is 1 for action 0, and 0 for all other actions.
- In the high reward condition, the characters want to eat the food together. So the reward is 0 for action 0, and 1 for all other actions.

In [4]:
@jax.jit
def get_reward(action, reward_condition): 
    low_reward = jnp.array([1, 1, 1, 1])
    high_reward = jnp.array([0, 2, 2, 2])
    which_reward = jnp.where(reward_condition == RewardConditions.LOW, low_reward, high_reward)
    return which_reward[action]


@jax.jit
def get_cost(action, intimacy):
    """
    This is a basic discomfort term
    The more intimate the relationship, the smaller the cost.
    Most formal relationship -> keep original risk value
    Most intimate relationship -> scale down risk value
    """
    formality = 1 - intimacy
    risk = get_risk(action)
    return formality * risk

Memo stuff

In [5]:
@memo
def vanilla_actor[
    action: actions,
    intimacy: IntimacyLevels,
    reward_condition: RewardConditions,
](
    alpha, w_r, w_c
):
    cast: [actor]
    actor: knows(intimacy)
    actor: knows(reward_condition)
    actor: chooses(
        action in actions,
        wpp=exp(
            alpha
            * (w_r * get_reward(action, reward_condition) - w_c * get_risk(action))
        ),
    )

    return Pr[actor.action == action]


@memo
def relationship_actor[
    action: actions,
    intimacy: IntimacyLevels,
    reward_condition: RewardConditions,
](
    alpha, w_r, w_c
):
    cast: [actor]
    actor: knows(intimacy)
    actor: knows(reward_condition)
    actor: chooses(
        action in actions,
        wpp=exp(
            alpha
            * (
                w_r * get_reward(action, reward_condition)
                - w_c * get_cost(action, intimacy)
            )
        ),
    )

    return Pr[actor.action == action]


@memo
def vanilla_observer[
    action: actions,
    intimacy: IntimacyLevels,
    reward_condition: RewardConditions,
](
    alpha, w_r, w_c
):
    cast: [actor, observer]

    observer: knows(reward_condition)
    observer: thinks[
        actor : knows(reward_condition),
        actor : chooses(intimacy in IntimacyLevels, wpp=1),  # uniform prior
        actor : chooses(
            action in actions,
            wpp=vanilla_actor[action, intimacy, reward_condition](alpha, w_r, w_c),
        ),
    ]

    observer: observes[actor.action] is action
    observer: chooses(
        intimacy in IntimacyLevels,
        wpp=E[actor.intimacy == intimacy],
    )

    return Pr[observer.intimacy == intimacy]


@memo
def relationship_observer[
    action: actions,
    intimacy: IntimacyLevels,
    reward_condition: RewardConditions,
](
    alpha, w_r, w_c
):
    cast: [actor, observer]

    observer: knows(reward_condition)
    observer: thinks[
        actor : knows(reward_condition),
        actor : chooses(intimacy in IntimacyLevels, wpp=1),  # uniform prior
        actor : chooses(
            action in actions,
            wpp=relationship_actor[action, intimacy, reward_condition](alpha, w_r, w_c),
        ),
    ]

    observer: observes[actor.action] is action
    observer: chooses(
        intimacy in IntimacyLevels,
        wpp=E[actor.intimacy == intimacy],
    )

    return Pr[observer.intimacy == intimacy]

## Generate predictions

### Actor predictions

In [6]:
params = {
    "alpha": 2,
    "w_r": 2,
    "w_c": 1,
}

output_mappings = {
    "LOW": "low",
    "HIGH": "high"
}

In [7]:
vanilla_result = vanilla_observer(
    alpha=params["alpha"], w_r=params["w_r"], w_c=params["w_c"], return_pandas=True
)
relationship_result = relationship_observer(
    alpha=params["alpha"], w_r=params["w_r"], w_c=params["w_c"], return_pandas=True
)

df_vanilla = (
    vanilla_result[1]
    .pandas.rename(
        columns={
            "reward_condition": "reward",
            "la_observer": "density",
        }
    )
    .replace(output_mappings)
)
df_vanilla["model"] = "vanilla"

df_relationship = (
    relationship_result[1]
    .pandas.rename(
        columns={
            "reward_condition": "reward",
            "ionship_observer": "density",
        }
    )
    .replace(output_mappings)
)
df_relationship["model"] = "relationship"

df_combined = pd.concat([df_vanilla, df_relationship])

# add alpha, w_r, w_c to the beginning of the dataframe
df_combined.insert(0, "w_c", params["w_c"])
df_combined.insert(0, "w_r", params["w_r"])
df_combined.insert(0, "alpha", params["alpha"])

# divide density by bin width
df_combined['intimacy'] = df_combined['intimacy'] * 100

In [8]:
df_combined

,alpha,w_r,w_c,action,intimacy,reward,density,model
0,2,2,1,0,0.000000,low,0.009901,vanilla
1,2,2,1,0,0.000000,high,0.009901,vanilla
2,2,2,1,0,1.000000,low,0.009901,vanilla
3,2,2,1,0,1.000000,high,0.009901,vanilla
4,2,2,1,0,2.000000,low,0.009901,vanilla
...,...,...,...,...,...,...,...,...
803,2,2,1,3,97.999996,high,0.026437,relationship
804,2,2,1,3,98.999995,low,0.030313,relationship
805,2,2,1,3,98.999995,high,0.026982,relationship
806,2,2,1,3,100.000000,low,0.031085,relationship


In [9]:
df_combined.to_csv("model_preds_inverse_planning_relationship.csv", index=False)

## Inferring reward from action, given intimacy

TODO: this is repeating a lot of code from before, and redifining the same functions... change l8r

In [10]:
class RelationshipConditions(IntEnum):
    ZERO = 0
    FIFTY = 1
    SEVENTY_FIVE = 2
    ONE_HUNDRED = 3

In [11]:
@jax.jit
def get_intimacy(relationship_condition):
    return jnp.array([0, 0.5, 0.75, 1])[relationship_condition]

@jax.jit
def get_risk(action):
    return jnp.array([0, 0, 1, 2])[action]

In [12]:
@jax.jit
def get_cost(action, relationship_condition):
    """
    This is a basic discomfort term
    The more intimate the relationship, the smaller the cost.
    Most formal relationship -> keep original risk value
    Most intimate relationship -> scale down risk value
    """
    intimacy = get_intimacy(relationship_condition)
    formality = 1 - intimacy
    risk = get_risk(action)
    return formality * risk

In [13]:
@memo
def vanilla_actor[
    action: actions,
    relationship_condition: RelationshipConditions,
    reward_condition: RewardConditions,
](
    alpha, w_r, w_c
):
    cast: [actor]
    actor: knows(relationship_condition)
    actor: knows(reward_condition)
    actor: chooses(
        action in actions,
        wpp=exp(
            alpha
            * (w_r * get_reward(action, reward_condition) - w_c * get_risk(action))
        ),
    )

    return Pr[actor.action == action]


@memo
def relationship_actor[
    action: actions,
    relationship_condition: RelationshipConditions,
    reward_condition: RewardConditions,
](
    alpha, w_r, w_c
):
    cast: [actor]
    actor: knows(relationship_condition)
    actor: knows(reward_condition)
    actor: chooses(
        action in actions,
        wpp=exp(
            alpha
            * (
                w_r * get_reward(action, reward_condition)
                - w_c * get_cost(action, relationship_condition)
            )
        ),
    )

    return Pr[actor.action == action]


@jax.jit
def get_reward_prior(reward_condition, relationship_condition):
    # prior on reward depends on relationship
    # intenum reward_condition: 0 = low, 1 = high
    return jnp.array([[0.4, 0.6], [0.4, 0.6], [0.3, 0.7], [0.25, 0.75]])[
        relationship_condition, reward_condition
    ]


@memo
def vanilla_observer_reward[
    action: actions,
    relationship_condition: RelationshipConditions,
    reward_condition: RewardConditions,
](
    alpha, w_r, w_c
):
    cast: [actor, observer]

    observer: knows(relationship_condition)
    observer: thinks[
        actor : knows(relationship_condition),
        actor : chooses(
            reward_condition in RewardConditions,
            wpp=get_reward_prior(reward_condition, relationship_condition),
        ),  # uniform prior
        actor : chooses(
            action in actions,
            wpp=vanilla_actor[action, relationship_condition, reward_condition](
                alpha, w_r, w_c
            ),
        ),
    ]

    observer: observes[actor.action] is action
    observer: chooses(
        reward_condition in RewardConditions,
        wpp=E[actor.reward_condition == reward_condition],
    )

    return Pr[observer.reward_condition == reward_condition]


@memo
def relationship_observer_reward[
    action: actions,
    relationship_condition: RelationshipConditions,
    reward_condition: RewardConditions,
](
    alpha, w_r, w_c
):
    cast: [actor, observer]

    observer: knows(relationship_condition)
    observer: thinks[
        actor : knows(relationship_condition),
        actor : chooses(
            reward_condition in RewardConditions,
            wpp=get_reward_prior(reward_condition, relationship_condition),
        ),  # uniform prior
        actor : chooses(
            action in actions,
            wpp=relationship_actor[action, relationship_condition, reward_condition](
                alpha, w_r, w_c
            ),
        ),
    ]

    observer: observes[actor.action] is action
    observer: chooses(
        reward_condition in RewardConditions,
        wpp=E[actor.reward_condition == reward_condition],
    )

    return Pr[observer.reward_condition == reward_condition]

In [14]:
params = {
    "alpha": 1.5,
    "w_r": 1,
    "w_c": 3,
}

output_mappings = {
    "ZERO": 0,
    "FIFTY": 50,
    "SEVENTY_FIVE": 75,
    "ONE_HUNDRED": 100,
    "LOW": "low",
    "HIGH": "high"
}

In [15]:
vanilla_result = vanilla_observer_reward(
    alpha=params["alpha"], w_r=params["w_r"], w_c=params["w_c"], return_pandas=True
)
relationship_result = relationship_observer_reward(
    alpha=params["alpha"], w_r=params["w_r"], w_c=params["w_c"], return_pandas=True
)

df_vanilla = (
    vanilla_result[1]
    .pandas.rename(
        columns={
            "reward_condition": "reward",
            "relationship_condition": "intimacy",
            "la_observer_reward": "p_reward",
        }
    )
    .replace(output_mappings)
)
df_vanilla["model"] = "vanilla"

df_relationship = (
    relationship_result[1]
    .pandas.rename(
        columns={
            "reward_condition": "reward",
            "relationship_condition": "intimacy",
            "ionship_observer_reward": "p_reward",
        }
    )
    .replace(output_mappings)
)
df_relationship["model"] = "relationship"

df_combined = pd.concat([df_vanilla, df_relationship])

# add alpha, w_r, w_c to the beginning of the dataframe
df_combined.insert(0, "w_c", params["w_c"])
df_combined.insert(0, "w_r", params["w_r"])
df_combined.insert(0, "alpha", params["alpha"])

In [16]:
df_combined.to_csv("model_preds_inverse_planning_reward.csv", index=False)